## 结果

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

### 十折交叉验证结果

In [3]:
models = ['xgboost', 'svm', 'randomforest', 'mlp', 'lr', 'lgbm', 'knn']
model_file = {'svm':'svm_clf_model_cv_auc.csv',
              'randomforest':'rf_model_cv_auc.csv',
              'mlp':'mlp_relu_0.01_constant_(128, 64, 32)_model_cv_auc.csv',
              'lr':'lr_clf_model_cv_auc.csv',
              'lgbm':'lgbm_clf_model_cv_auc.csv',
              'knn':'knn_clf_model_cv_auc.csv'
             }
model_label = {'svm':'SVM', 
               'randomforest':'RandomForest', 
               'mlp':'MLP', 
               'lr':'LogisticRegression', 
               'lgbm':'LightGBM', 
               'knn':'KNN'
              }

cv_results = pd.DataFrame()
for model in models:
    filepath = f'../results/{model}/cv/'
    if model == 'xgboost':
        res = pd.read_csv(f'{filepath}XGBoost_model_withnan_cv_auc.csv')
        res['model'] = 'XGBoost with NaN'
        cv_results = pd.concat([cv_results, res])
        
        res = pd.read_csv(f'{filepath}XGBoost_model_cv_auc.csv')
        res['model'] = 'XGBoost'
        cv_results = pd.concat([cv_results, res])
    else:
        res = pd.read_csv(f'{filepath}{model_file[model]}')
        res['model'] = model_label[model]
        cv_results = pd.concat([cv_results, res])

In [4]:
print(cv_results.shape)
print(cv_results.head())

(80, 7)
   fold       AUC  accuracy  precision    recall        F1             model
0     1  0.988248  0.953930   0.953488  0.947977  0.950725  XGBoost with NaN
1     2  0.977350  0.934959   0.940828  0.919075  0.929825  XGBoost with NaN
2     3  0.980919  0.959350   0.970238  0.942197  0.956012  XGBoost with NaN
3     4  0.988587  0.951220   0.953216  0.942197  0.947674  XGBoost with NaN
4     5  0.983774  0.953804   0.964072  0.936047  0.949853  XGBoost with NaN


In [5]:
grouped = cv_results.groupby('model')['AUC']
mean_auc = grouped.mean().reset_index(name='mean_auc').sort_values(by='mean_auc', ascending=True)
model_order = mean_auc.model.to_list()
print('model order:', model_order)

# Init a figure and axes
fig, ax = plt.subplots(figsize=(8, 6))

# Create the plot with different colors for each group
boxplot = ax.boxplot(x=[grouped.get_group(g) for g in model_order],
                     labels=model_order,
                     patch_artist=True,
                     medianprops={'color': 'black'},
                     vert=False
                    ) 
# 设置统一颜色
for patch in boxplot['boxes']:
    patch.set_facecolor('#69b3a2')

ax.set_xlabel('AUCs (10-fold cross validation)')    

plt.tight_layout()   # 避免标签被裁剪
plt.savefig("../results/figs/cv_performance.png", dpi=300, bbox_inches='tight')
plt.close()          # 可选：避免在循环中重复显示

model order: ['KNN', 'SVM', 'MLP', 'LogisticRegression', 'RandomForest', 'XGBoost', 'LightGBM', 'XGBoost with NaN']


### 测试集结果

In [6]:
models = ['xgboost', 'svm', 'randomforest', 'mlp', 'lr', 'lgbm', 'knn']
model_file = {'svm':'svm_clf_model.csv',
              'randomforest':'rf_clf_model.csv',
              'mlp':'mlp_model.csv',
              'lr':'lr_clf_model.csv',
              'lgbm':'lgbm_clf_model.csv',
              'knn':'knn_clf_model.csv'
             }
model_label = {'svm':'SVM', 
               'randomforest':'RandomForest', 
               'mlp':'MLP', 
               'lr':'LogisticRegression', 
               'lgbm':'LightGBM', 
               'knn':'KNN'
              }

test_results = pd.DataFrame()
for model in models:
    filepath = f'../results/{model}/test/'
    if model == 'xgboost':
        res = pd.read_csv(f'{filepath}xgb_clf_model_withnan.csv')
        res['Model'] = 'XGBoost with NaN'
        test_results = pd.concat([test_results, res])
        
        res = pd.read_csv(f'{filepath}xgb_clf_model.csv')
        res['Model'] = 'XGBoost'
        test_results = pd.concat([test_results, res])
    else:
        res = pd.read_csv(f'{filepath}{model_file[model]}')
        res['Model'] = model_label[model]
        test_results = pd.concat([test_results, res])
test_results.sort_values(by='AUC', ascending=False, inplace=True)

In [7]:
print(test_results.shape)
print(test_results)

(8, 6)
   Accuracy  Precision    Recall        F1       AUC               Model
0  0.954397   0.964200  0.937355  0.950588  0.990151    XGBoost with NaN
0  0.903366   0.923267  0.865429  0.893413  0.966918            LightGBM
0  0.902280   0.904988  0.883991  0.894366  0.962361             XGBoost
0  0.889251   0.904177  0.853828  0.878282  0.944453        RandomForest
0  0.798046   0.813299  0.737819  0.773723  0.865950  LogisticRegression
0  0.777416   0.842424  0.645012  0.730618  0.835778                 MLP
0  0.666667   0.685629  0.531323  0.598693  0.733283                 SVM
0  0.675353   0.695266  0.545244  0.611183  0.701394                 KNN


In [8]:
test_results.to_csv('../results/test_results.csv', index=False)

In [9]:
import plotly.graph_objects as go
import pandas as pd
from math import pi

# 模型性能数据
df = test_results.reset_index(drop=True).copy()
categories = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']

# number of variable
N = len(categories)
 
# What will be the angle of each axis in the plot? (we divide the plot / number of variable)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': 'polar'})
 
# If you want the first axis to be on top:
ax.set_theta_offset(pi / 2)
ax.set_theta_direction(-1)
 
# Draw one axe per variable + add labels
plt.xticks(angles[:-1], categories, size=12)
 
# Draw ylabels
ax.set_rlabel_position(0)
plt.yticks([0.4,0.5,0.6,0.7,0.8,0.9,1], ["0.40","0.50","0.60","0.70","0.80","0.90","1.00"], color="black", size=9)
plt.ylim(0.4,1)
 

# ------- PART 2: Add plots
 
# Plot each individual = each line of the data
# I don't make a loop, because plotting more than 3 groups makes the chart unreadable
colors = ['#729ECE', '#FF9E4A', '#67BF5C', '#ED665D', '#AD8BC9', '#ED97CA', '#CDCC5D', '#6DCCDA']
for i in range(len(df)):
    values=df.loc[i].drop('Model').values.flatten().tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=1, linestyle='solid', label=df.loc[i, 'Model'], color=colors[i])
    ax.fill(angles, values, colors[i], alpha=0.1)

plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))

# Show the graph
plt.tight_layout()   # 避免标签被裁剪
plt.savefig("../results/figs/test_performance.png", dpi=300, bbox_inches='tight')
plt.close()          # 可选：避免在循环中重复显示

In [10]:
def make_spider(row, title, color):

    # number of variable
    N = len(categories)

    # What will be the angle of each axis in the plot? (we divide the plot / number of variable)
    angles = [n / float(N) * 2 * pi for n in range(N)]
    angles += angles[:1]

    # Initialise the spider plot
    ax = plt.subplot(2,4,row+1, polar=True, )

    # If you want the first axis to be on top:
    ax.set_theta_offset(pi / 2)
    ax.set_theta_direction(-1)

    # Draw one axe per variable + add labels labels yet
    plt.xticks(angles[:-1], categories, color='black', size=9)

    # Draw ylabels
    ax.set_rlabel_position(0)
    plt.yticks([0.4,0.5,0.6,0.7,0.8,0.9,1], ["0.40","0.50","0.60","0.70","0.80","0.90","1.00"], color="black", size=8)
    plt.ylim(0.4,1)

    # Ind1
    values=df.loc[row].drop('Model').values.flatten().tolist()
    values += values[:1]
    ax.plot(angles, values, color=color, linewidth=1, linestyle='solid')
    ax.fill(angles, values, color=color, alpha=0.4)

    # Add a title
    plt.title(title, size=11, color=color, y=1.1)

    
# ------- PART 2: Apply the function to all individuals
# initialize the figure
plt.figure(figsize=(20,10)) 
# Loop to plot
for i in range(len(df)):
    make_spider( row=i, title=df.loc[i, 'Model'], color=colors[i])
    
plt.tight_layout()   # 避免标签被裁剪
plt.savefig("../results/figs/test_performance_ind.png", dpi=300, bbox_inches='tight')
plt.close()          # 可选：避免在循环中重复显示